# Wan 2.2 Video Generation API
**BlueRidgeCustomCo — TikTok Product Video Generator**

This notebook runs Wan 2.2 image-to-video on Colab's free GPU and exposes a Gradio API endpoint your VPS can call.

## Setup
1. Go to **Runtime → Change runtime type → T4 GPU** (or A100 if available)
2. Run all cells in order
3. Copy the Gradio public URL from the last cell output
4. Paste it into your vinylApp `.env` as `WAN_VIDEO_API_URL`

In [ ]:
#@title 1. Install Dependencies & Clone Wan2.2
!pip install -q gradio pillow
!git clone https://github.com/Wan-Video/Wan2.2.git /content/wan2.2
%cd /content/wan2.2
!pip install -q -r requirements.txt
!pip install -q huggingface_hub
print('\n✅ Dependencies installed')

In [ ]:
#@title 2. Download Wan2.2 Image-to-Video Model (1.3B — fast, fits free tier)
import os
from huggingface_hub import snapshot_download

# Use 1.3B for free T4 tier (faster, lower VRAM)
# Switch to "Wan-AI/Wan2.2-I2V-14B-480P" if you get A100
MODEL_ID = "Wan-AI/Wan2.2-I2V-1.3B-480P"  #@param ["Wan-AI/Wan2.2-I2V-1.3B-480P", "Wan-AI/Wan2.2-I2V-14B-480P"]
MODEL_DIR = f"/content/models/{MODEL_ID.split('/')[-1]}"

if not os.path.exists(MODEL_DIR):
    print(f'Downloading {MODEL_ID}...')
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False
    )
    print(f'✅ Model downloaded to {MODEL_DIR}')
else:
    print(f'✅ Model already exists at {MODEL_DIR}')

# Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
#@title 3. Launch Gradio Video Generation API
import gradio as gr
import subprocess
import glob
import shutil
import tempfile
import time
import uuid
import json
from pathlib import Path
from PIL import Image

OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_video(
    image,
    prompt="The subject gently animates with subtle movement, slight rotation, and a soft zoom effect.",
    resolution="832*480",
    num_frames=81,
    sample_steps=20
):
    """Generate a video from an input image using Wan2.2"""
    start = time.time()
    job_id = str(uuid.uuid4())[:8]
    work_dir = f"/content/jobs/{job_id}"
    os.makedirs(work_dir, exist_ok=True)

    # Save input image
    input_path = f"{work_dir}/input.png"
    if isinstance(image, str):
        shutil.copy(image, input_path)
    else:
        Image.fromarray(image).save(input_path)

    # Determine task based on model
    task = "i2v-1.3B" if "1.3B" in MODEL_DIR else "i2v-14B"

    # Build command
    cmd = [
        "python", "generate.py",
        "--task", task,
        "--size", resolution,
        "--ckpt_dir", MODEL_DIR,
        "--image", input_path,
        "--prompt", prompt,
        "--frame_num", str(num_frames),
        "--sample_steps", str(sample_steps),
        "--offload_model", "true",
        "--use_t5_cpu",
        "--save_file", f"{work_dir}/output.mp4"
    ]

    print(f"[{job_id}] Generating video...")
    print(f"  Prompt: {prompt}")
    print(f"  Resolution: {resolution}, Frames: {num_frames}")

    result = subprocess.run(
        cmd,
        cwd="/content/wan2.2",
        capture_output=True,
        text=True,
        timeout=1800
    )

    if result.returncode != 0:
        print(f"STDERR: {result.stderr[-500:]}")
        # Try to find any mp4 output
        mp4s = glob.glob(f"{work_dir}/*.mp4") + glob.glob("/content/wan2.2/*.mp4")
        if not mp4s:
            return None
        output_path = mp4s[0]
    else:
        output_path = f"{work_dir}/output.mp4"
        if not os.path.exists(output_path):
            mp4s = glob.glob(f"{work_dir}/*.mp4") + glob.glob("/content/wan2.2/*.mp4")
            output_path = mp4s[0] if mp4s else None

    if output_path and os.path.exists(output_path):
        # Copy to outputs with timestamp
        final_path = f"{OUTPUT_DIR}/{job_id}.mp4"
        shutil.copy(output_path, final_path)
        elapsed = time.time() - start
        print(f"[{job_id}] ✅ Done in {elapsed:.0f}s — {final_path}")
        return final_path

    print(f"[{job_id}] ❌ No output video found")
    return None


def health_check():
    """Health check endpoint"""
    return json.dumps({
        "status": "ok",
        "model": MODEL_DIR.split("/")[-1],
        "gpu": subprocess.getoutput("nvidia-smi --query-gpu=name,memory.free --format=csv,noheader")
    })


# Build Gradio interface
with gr.Blocks(title="Wan2.2 Video API") as app:
    gr.Markdown("## Wan 2.2 — Image to Video API")
    gr.Markdown("Upload a product image and describe the animation you want.")

    with gr.Row():
        with gr.Column():
            img_input = gr.Image(label="Input Image", type="numpy")
            prompt_input = gr.Textbox(
                label="Motion Prompt",
                value="The product gently rotates with a subtle zoom, showcasing details from multiple angles.",
                lines=3
            )
            resolution_input = gr.Dropdown(
                choices=["832*480", "1280*720", "480*832"],
                value="832*480",
                label="Resolution"
            )
            frames_input = gr.Slider(
                minimum=41, maximum=81, step=8, value=81,
                label="Frames (81≈5sec, 41≈2.5sec)"
            )
            steps_input = gr.Slider(
                minimum=10, maximum=30, step=5, value=20,
                label="Sample Steps (lower=faster, lower quality)"
            )
            gen_btn = gr.Button("Generate Video", variant="primary")
        with gr.Column():
            video_output = gr.Video(label="Generated Video")

    gen_btn.click(
        fn=generate_video,
        inputs=[img_input, prompt_input, resolution_input, frames_input, steps_input],
        outputs=video_output
    )

    # Health check API
    health_btn = gr.Button("Health Check", visible=False)
    health_output = gr.Textbox(visible=False)
    health_btn.click(fn=health_check, outputs=health_output)

print("\n" + "="*60)
print("LAUNCHING GRADIO API")
print("Copy the public URL below and paste it into your .env")
print("as: WAN_VIDEO_API_URL=https://xxxxx.gradio.live")
print("="*60 + "\n")

app.launch(share=True, show_error=True)